### function

In [1]:
import numpy as np
from sklearn.manifold import Isomap
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import shortest_path
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
import time

def compute_geodesic_distances_isomap(X, n_neighbors=15, n_components=10, sample_fraction=0.05):
    """
    Compute approximate geodesic distance matrix for high-dimensional data using Isomap.
    
    Parameters:
        X: (n_samples, n_features) data matrix
        n_neighbors: number of k-nearest neighbors
        n_components: Isomap embedding dimension (for acceleration)
        sample_fraction: subsampling ratio (0~1), use 0.05~0.1 if memory is limited
    
    Returns:
        geodesic_dist: (n_samples_sub, n_samples_sub) geodesic distance matrix
        sample_idx: original indices of the subsampled points
    """
    if sample_fraction < 1.0:
        n_samples = X.shape[0]
        n_sub = int(n_samples * sample_fraction)
        np.random.seed(42)
        sample_idx = np.random.choice(n_samples, n_sub, replace=False)
        X_sub = X[sample_idx, :]
        print(f"Subsampled to {n_sub} samples")
    else:
        X_sub = X
        sample_idx = np.arange(X.shape[0])
        n_sub = X_sub.shape[0]
    
    # Use Isomap to approximate geodesic distances
    # Isomap builds a k-NN graph internally, then computes shortest paths
    print("Building Isomap model...")
    start_time = time.time()
    
    isomap = Isomap(
        n_neighbors=n_neighbors,
        n_components=n_components,
        metric='euclidean',
        n_jobs=-1
    )
    embedding = isomap.fit_transform(X_sub)
    
    # distance_matrix_ attribute contains geodesic distances
    if hasattr(isomap, 'distance_matrix_'):
        geodesic_dist = isomap.distance_matrix_
    else:
        # Fallback: manually compute shortest paths from k-NN graph
        from sklearn.neighbors import kneighbors_graph
        knn_graph = kneighbors_graph(X_sub, n_neighbors=n_neighbors, mode='distance', include_self=False)
        geodesic_dist = shortest_path(csgraph=knn_graph, directed=False, method='D')
    
    elapsed = time.time() - start_time
    print(f"Geodesic distance computation completed in {elapsed:.2f} seconds")
    print(f"Distance matrix shape: {geodesic_dist.shape}")
    
    return geodesic_dist, sample_idx




In [10]:
import numpy as np
import scanpy as sc
import pandas as pd
sc_data_path = "/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_Xenium/adata_sc_Xenium.h5ad"
import scanpy as sc
import numpy as np
adata = sc.read_h5ad(sc_data_path)
import scanpy as sc



In [11]:
sc.pp.filter_genes(adata, min_cells=1)
sc.pp.filter_cells(adata, min_genes=1)
adata.var['mt'] = adata.var_names.str.startswith('mt-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
adata = adata[adata.obs.pct_counts_mt < 20, :]
print(f"filtered: {adata.shape[0]} cells × {adata.shape[1]} genes")

filtered: 16068 cells × 313 genes


In [ ]:
sc.pp.normalize_total(adata, target_sum=1e2)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,          # 常用 2000 或 3000
    batch_key=None,
    flavor='seurat_v3',        # 推荐使用 seurat_v3 算法
    subset=True                # 直接保留高变基因子集
)

print(f"高变基因筛选后: {adata.shape[0]} cells × {adata.shape[1]} genes")

# ----------------- 4. 数据缩放（PCA 前必需） -----------------
# 将每个基因缩放到单位方差（mean=0, var=1）
sc.pp.scale(adata, max_value=10)   # max_value 截断，防止极端值影响 PCA

# ----------------- 5. 计算 PCA (100 维) -----------------
# run PCA
sc.tl.pca(
    adata, 
    svd_solver='arpack',      # 对于大规模数据，arpack 更快
    n_comps=100,              # 你要求的 100 维
    zero_center=True,         # 已经 scale 过了，但显式指定
    use_highly_variable=True, # 默认 True，使用高变基因
    random_state=42     )

/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/site-packages/scanpy/preprocessing/_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [5]:
pca_scores = adata.obsm['X_pca']   # (n_cells, 100)
pca_loadings = adata.varm['PCs']   # (n_genes, 100)

In [16]:
# ----------------- Usage Example -----------------
# X is your data matrix (19979, 1605)
# Subsampling is recommended due to large sample size
geodesic_dist, sample_idx = compute_geodesic_distances_isomap(
    pca_scores, 
    n_neighbors=20,
    n_components=30,
    sample_fraction=0.25
)

print(f"Geodesic distance - Mean: {geodesic_dist.mean():.4f}, Std: {geodesic_dist.std():.4f}")

Subsampled to 4017 samples
Building Isomap model...
Geodesic distance computation completed in 9.89 seconds
Distance matrix shape: (4017, 4017)
Geodesic distance - Mean: 39.1372, Std: 11.8008


In [17]:
np.save('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/Ours/geodesic_dist_sub.npy',geodesic_dist ,allow_pickle=True)


In [ ]:
np.save('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/Ours/sample_idx.npy',sample_idx,allow_pickle=True)


: 